# Lecture 09 — Inferencia con Modelos Preentrenados: YOLO

**Introduccion a la Inteligencia Artificial — EAFIT 2026-01**

---

En esta clase vamos a trabajar con un modelo de IA **preentrenado** para realizar inferencia sobre imagenes. El modelo que usaremos es **YOLO (You Only Look Once)**, uno de los detectores de objetos mas populares y eficientes del mundo.

### ¿Que aprenderemos hoy?

| Nivel | Descripcion | Herramientas |
|-------|-------------|--------------|
| **Nivel 1** | Inferencia directa con YOLO en el notebook | Python, Ultralytics |
| **Nivel 2** | YOLO como API REST local | Docker, FastAPI |

> En esta sesion cubrimos el **Nivel 1** completo.

### Concepto clave: ¿Que es inferencia?

**Inferencia** es el proceso de usar un modelo ya entrenado para hacer predicciones sobre datos nuevos.  
No entrenamos nada — simplemente usamos el conocimiento que el modelo ya adquirio.

## 1. ¿Que es YOLO?

YOLO es una **red neuronal convolucional** disenada para detectar objetos en imagenes en tiempo real.

### ¿Como funciona?

A diferencia de enfoques anteriores que analizaban una imagen en multiples pasadas, YOLO divide la imagen en una cuadricula y **en una sola pasada** predice simultaneamente:

```
Imagen de entrada
      |
      v
┌─────────────────────────────────────────┐
│         Red Neuronal YOLO               │
│   (procesamiento en una sola pasada)    │
└─────────────────────────────────────────┘
      |
      v
Para cada region de la imagen:
  - ¿Que objeto hay?     → clase (persona, carro, perro...)
  - ¿Donde esta?         → bounding box (x, y, ancho, alto)
  - ¿Que tan seguro?     → confidence score (0.0 a 1.0)
```

### Modelos disponibles (familia YOLO26 — enero 2026)

| Modelo | Parametros | mAP COCO | Velocidad | Uso recomendado |
|--------|-----------|----------|-----------|-----------------|
| `yolo26n.pt` | 2.4 M | 40.9 | Muy rapido | Prototipado, edge devices |
| `yolo26s.pt` | 9.5 M | 48.6 | Rapido | Aplicaciones moviles |
| `yolo26m.pt` | 20.4 M | 53.1 | Moderado | Uso general |
| `yolo26l.pt` | 24.8 M | 55.0 | Lento | Produccion |
| `yolo26x.pt` | 55.7 M | 57.5 | Muy lento | Investigacion |

> En esta clase usaremos **`yolo26n.pt`** (nano) — 2.4 millones de parametros, suficiente para aprender los conceptos.

### Novedad de YOLO26: sin NMS

Versiones anteriores necesitaban un paso post-procesamiento llamado **NMS (Non-Maximum Suppression)** para eliminar detecciones duplicadas. YOLO26 incorpora ese filtrado **dentro de la red neuronal**, simplificando el despliegue.

### Dataset de entrenamiento: COCO

YOLO fue preentrenado con el dataset **COCO (Common Objects in Context)** que contiene:
- ~330,000 imagenes
- 80 clases de objetos cotidianos (personas, vehiculos, animales, muebles, etc.)

## 2. Instalacion

Solo necesitamos una libreria: **`ultralytics`**, que incluye YOLO y todas sus dependencias.

In [ ]:
# Instalación original, opcional para no reinstalar paquetes en cada ejecución.
INSTALAR_DEPENDENCIAS = False
EJECUTAR_INFERENCIA = False
DESCARGAR_IMAGEN_EJEMPLO = False
MODELO = "yolo26n.pt"
UMBRAL_IMAGEN_PROPIA = 0.50
MI_IMAGEN = None  # Path local o None para selector Colab / pregunta de ruta.
if INSTALAR_DEPENDENCIAS:
    get_ipython().run_line_magic("pip", "install -q -U ultralytics")
else:
    print("Instalación desactivada. Activa EJECUTAR_INFERENCIA para ejecutar el ejemplo.")

## 3. Cargando el modelo

Al ejecutar `YOLO("yolo26n.pt")` por primera vez, el modelo se **descarga automaticamente** desde los servidores de Ultralytics y se guarda localmente para usos futuros.

> `yolo26n.pt` tiene **2.4 millones de parametros** — cada uno es un numero que la red aprendio durante el entrenamiento con millones de imagenes.

In [ ]:
model = None
if EJECUTAR_INFERENCIA:
    from ultralytics import YOLO
    model = YOLO(MODELO)  # Descarga los pesos solo si no existen localmente.
    print(f"Modelo cargado: {type(model)}")
    print(f"Numero de clases que puede detectar: {len(model.names)}")
    print(f"Primeras 5 clases: {list(model.names.values())[:5]}")
else:
    print("Carga de pesos e inferencia desactivadas.")

In [ ]:
if model is not None:
    print("Todas las clases del dataset COCO:\n")
    for clase_id, nombre in model.names.items():
        print(f"  {clase_id:3d}: {nombre}")

## 4. Inferencia sobre una imagen

Vamos a descargar una imagen de prueba y ejecutar YOLO sobre ella.

YOLO acepta como entrada:
- Una ruta a un archivo de imagen (`"foto.jpg"`)
- Una URL directa a una imagen
- Un array NumPy
- Un objeto PIL Image

In [ ]:
import urllib.request
from pathlib import Path
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

url = "https://ultralytics.com/images/bus.jpg"
imagen_path = Path("images/bus.jpg")
img = None
if EJECUTAR_INFERENCIA:
    if not imagen_path.exists() and DESCARGAR_IMAGEN_EJEMPLO:
        imagen_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, imagen_path)
    if imagen_path.is_file():
        with Image.open(imagen_path) as original:
            img = ImageOps.exif_transpose(original).convert("RGB")
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.title("Imagen original — ejemplo del material")
        plt.axis("off")
        plt.show()
        print(f"Tamano de la imagen: {img.size[0]}x{img.size[1]} pixeles")
    else:
        print("Ejemplo bus.jpg omitido: permite su descarga o continúa con tu propia imagen.")

In [ ]:
resultados = []
if model is not None and img is not None:
    resultados = model(img, verbose=False)
    print(f"Tipo del resultado: {type(resultados)}")
    print(f"Numero de imagenes procesadas: {len(resultados)}")

## 5. Explorando el objeto de resultados

YOLO retorna una lista de objetos `Results`. Cada uno contiene toda la informacion de las detecciones para esa imagen.

In [ ]:
if resultados:
    r = resultados[0]
    print("=== Informacion general ===")
    print(f"Imagen original: {r.path}")
    print(f"Tamano procesado: {r.orig_shape}")
    print(f"Objetos detectados: {len(r.boxes)}")
    print("\n=== Detecciones ===")
    for i, box in enumerate(r.boxes):
        clase_id = int(box.cls[0])
        clase_nom = model.names[clase_id]
        confianza = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f"[{i+1}] {clase_nom:15s} | confianza: {confianza:.2f} | "
              f"bbox: ({x1:.0f},{y1:.0f}) → ({x2:.0f},{y2:.0f})")
else:
    print("No hay resultados del ejemplo; no se han simulado detecciones.")

## 6. Visualizando las detecciones

YOLO puede generar automaticamente la imagen anotada con los bounding boxes y etiquetas.

### Conversion BGR a RGB

`r.plot()` retorna la imagen usando **OpenCV** internamente, que historicamente guarda los canales de color en orden **BGR** (Azul, Verde, Rojo). Matplotlib en cambio espera orden **RGB** (Rojo, Verde, Azul).

```
OpenCV / YOLO         Matplotlib
canal 0 = Azul   →   canal 0 = Rojo
canal 1 = Verde  →   canal 1 = Verde
canal 2 = Rojo   →   canal 2 = Azul
```

Si mostramos la imagen sin convertir, los colores quedan **invertidos** (lo rojo aparece azul y viceversa).

La conversion se hace con slicing de NumPy:

```python
imagen_anotada_rgb = imagen_anotada[:, :, ::-1]
#                                   │    │   └── invierte el orden de los 3 canales
#                                   │    └────── todos los pixeles del ancho
#                                   └──────────── todos los pixeles del alto
```

El `::-1` recorre el eje de canales al reves: `[0

In [ ]:
import numpy as np
if resultados:
    imagen_anotada = resultados[0].plot()
    imagen_anotada_rgb = imagen_anotada[:, :, ::-1]
    plt.figure(figsize=(12, 7))
    plt.imshow(imagen_anotada_rgb)
    plt.title(f"YOLO — {len(resultados[0].boxes)} objetos detectados")
    plt.axis("off")
    plt.show()

## 7. Ajustando el umbral de confianza

El parametro `conf` controla que tan "seguro" debe estar YOLO para reportar una deteccion.

- `conf=0.25` → reporta detecciones con 25%+ de confianza (mas detecciones, mas ruido)
- `conf=0.75` → solo detecciones muy seguras (menos detecciones, mas precision)

Experimenta con diferentes valores para ver como afecta los resultados:

In [ ]:
umbrales = [0.25, 0.50, 0.75]
if model is not None and img is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, umbral in zip(axes, umbrales):
        r = model(img, conf=umbral, verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"conf={umbral} → {len(r.boxes)} detecciones")
        ax.axis("off")
    plt.suptitle("Efecto del umbral de confianza")
    plt.tight_layout()
    plt.show()

## 8. Ejercicio — Prueba con tu propia imagen

Sube cualquier imagen y ejecuta YOLO sobre ella. Puedes usar una foto tuya, de tu escritorio, o descargar cualquier imagen de internet.

In [ ]:
# EJERCICIO RESUELTO: seleccionar una imagen y usar un umbral numérico.
from io import BytesIO
imagen_propia = None
resultado_propio = None
if EJECUTAR_INFERENCIA:
    if MI_IMAGEN is None:
        try:
            from google.colab import files
        except ImportError:
            texto = input("Ruta a tu imagen: ").strip().strip('"')
            if not texto:
                raise ValueError("Se necesita una imagen para completar el ejercicio.")
            ruta_propia = Path(texto).expanduser()
            contenido_propio = ruta_propia.read_bytes()
            nombre_propio = ruta_propia.name
        else:
            subidas = files.upload()
            if len(subidas) != 1:
                raise ValueError("Selecciona exactamente una imagen.")
            nombre_propio, contenido_propio = next(iter(subidas.items()))
            nombre_propio = Path(nombre_propio).name
    else:
        ruta_propia = Path(MI_IMAGEN).expanduser()
        contenido_propio = ruta_propia.read_bytes()
        nombre_propio = ruta_propia.name
    with Image.open(BytesIO(contenido_propio)) as original:
        imagen_propia = ImageOps.exif_transpose(original).convert("RGB")
    resultado_propio = model(imagen_propia, conf=UMBRAL_IMAGEN_PROPIA, verbose=False)[0]
    r = resultado_propio
    img_anotada = r.plot()[:, :, ::-1]
    plt.figure(figsize=(12, 7))
    plt.imshow(img_anotada)
    plt.title(f"Tu imagen — {len(r.boxes)} objetos detectados")
    plt.axis("off")
    plt.show()
    print("\nDetecciones:")
    for box in r.boxes:
        clase = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        print(f"  - {clase} ({conf:.0%} de confianza)")
else:
    print("Ejercicio listo. Activa EJECUTAR_INFERENCIA y proporciona tu imagen.")

### Respuesta y comparación sobre la imagen elegida

El espacio del umbral quedó resuelto con `0.50`, configurable. La ruta se obtiene por carga o selección local. La siguiente ampliación aplica los tres umbrales originales a la misma imagen y exporta resultados reales. Una puntuación alta no demuestra acierto: subir el umbral puede reducir falsas alarmas y también omitir objetos; la precisión requiere anotaciones reales.

In [ ]:
import csv
import json
from collections import Counter
import numpy as np

def detecciones_a_filas(resultado, umbral):
    """Serializa cajas de Ultralytics; también maneja cero detecciones."""
    if not np.isfinite(umbral) or not 0 <= umbral <= 1:
        raise ValueError("El umbral debe estar entre 0 y 1.")
    if resultado.boxes is None:
        return []
    cajas = resultado.boxes
    xyxy = cajas.xyxy.cpu().numpy()
    confianzas = cajas.conf.cpu().numpy()
    clases = cajas.cls.cpu().numpy().astype(int)
    if xyxy.shape != (len(confianzas), 4) or len(clases) != len(confianzas):
        raise ValueError("Cajas, confianzas y clases no están alineadas.")
    if not np.isfinite(xyxy).all() or not np.isfinite(confianzas).all():
        raise ValueError("Hay valores no finitos en las detecciones.")
    filas = []
    for caja, confianza, clase in zip(xyxy, confianzas, clases):
        if confianza >= umbral:
            filas.append({"umbral": float(umbral), "clase_id": int(clase),
                "clase": str(resultado.names[int(clase)]), "confianza": float(confianza),
                **dict(zip(("x1", "y1", "x2", "y2"), map(float, caja)))})
    return filas

def exportar_detecciones(ruta, filas):
    columnas = ("umbral", "clase_id", "clase", "confianza", "x1", "y1", "x2", "y2")
    with ruta.open("w", newline="", encoding="utf-8-sig") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=columnas)
        escritor.writeheader()
        escritor.writerows(filas)

def contar_clases(filas):
    return dict(sorted(Counter(fila["clase"] for fila in filas).items()))

In [ ]:
# Datos sintéticos: prueban la serialización, no el rendimiento de YOLO.
from types import SimpleNamespace

class TensorSimulado:
    def __init__(self, datos):
        self.datos = np.asarray(datos)
    def cpu(self):
        return self
    def numpy(self):
        return self.datos

simulado = SimpleNamespace(names={0: "A", 1: "B"}, boxes=SimpleNamespace(
    xyxy=TensorSimulado([[1, 2, 11, 12], [20, 21, 30, 31], [3, 4, 7, 8]]),
    conf=TensorSimulado([0.25, 0.50, 0.90]), cls=TensorSimulado([0, 1, 0]),
))
assert [len(detecciones_a_filas(simulado, u)) for u in (0.25, 0.50, 0.75)] == [3, 2, 1]
assert contar_clases(detecciones_a_filas(simulado, 0.25)) == {"A": 2, "B": 1}
assert detecciones_a_filas(SimpleNamespace(boxes=None), 0.25) == []
assert json.loads(json.dumps(detecciones_a_filas(simulado, 0.50)))[0]["clase"] == "B"
for umbral_invalido in (-1, 2, float("nan")):
    try:
        detecciones_a_filas(simulado, umbral_invalido)
    except ValueError:
        pass
    else:
        raise AssertionError("Se aceptó un umbral inválido.")
print("Pruebas sintéticas del esquema y los umbrales: OK. Sin inferencia real.")

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib

if imagen_propia is not None:
    salida = Path("resultados_yolo_intro") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    salida.mkdir(parents=True, exist_ok=False)
    filas, resumen = [], []
    figura, ejes = plt.subplots(1, 3, figsize=(18, 7))
    for eje, umbral in zip(ejes, umbrales):
        res = model(imagen_propia, conf=umbral, verbose=False)[0]
        detecciones = detecciones_a_filas(res, umbral)
        filas.extend(detecciones)
        resumen.append({"umbral": umbral, "detecciones": len(detecciones),
                        "conteos_por_clase": contar_clases(detecciones)})
        rgb = res.plot()[:, :, ::-1].copy()
        Image.fromarray(rgb).save(salida / f"conf_{umbral:.2f}.png")
        eje.imshow(rgb)
        eje.set_title(f"conf={umbral:.2f}: {len(detecciones)} detecciones")
        eje.axis("off")
        print(resumen[-1])
    figura.tight_layout()
    figura.savefig(salida / "comparacion.png", dpi=150, bbox_inches="tight")
    plt.show()
    exportar_detecciones(salida / "detecciones.csv", filas)
    observacion = input("Observación visual de cajas, omisiones o errores (Enter: no registrada): ").strip()
    informe = {"estado": "inferencia ejecutada", "imagen": nombre_propio,
        "sha256": hashlib.sha256(contenido_propio).hexdigest(), "modelo": MODELO,
        "ultralytics": version("ultralytics"), "resumen": resumen, "detecciones": filas,
        "observacion_visual": observacion or "No registrada", "metricas_con_etiquetas_reales": None}
    (salida / "informe.json").write_text(
        json.dumps(informe, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    print("Resultados:", salida.resolve())
else:
    print("No se exportan resultados: todavía no se ha procesado una imagen propia.")

### Cambios realizados y estado real

Se conservan todas las celdas Markdown originales, sus IDs y su orden. Se completó la celda del ejercicio con selección de imagen y confianza numérica; se añadieron controles para instalación/descarga, exportación y pruebas sintéticas. La explicación original sobre canales queda intacta; la inversión BGR→RGB se implementa con `[:, :, ::-1]`.

**Validado:** estructura del notebook, compilación y pruebas sintéticas. **Pendiente de evidencia personal:** ejecutar con tu imagen y revisar sus detecciones. No se entrenó ni se ejecutó inferencia real durante esta preparación.

Fuente fijada: [notebook original](https://github.com/EAFIT-IA/si3003-artificial-intelligence/blob/dfbcd874eb8bc27a3f3c258b0e7fdc41a6d9dbd3/notebooks/lecture8/yolo/Lecture_08_Yolo_Intro.ipynb).